In [1]:
from pymongo import MongoClient

# 1. Conectar al contenedor de Mongo
client = MongoClient('mongodb://mongodb:27017/')

# 2. Crear (o usar) la base de datos y la colección
db = client['TiendaBigData'] 
coleccion = db['Agrotech'] # Ideal el nombre del grupo ejem: 'G_Ecommerce_scraping'

print("Conexión exitosa a MongoDB")

Conexión exitosa a MongoDB


In [12]:
# --- PASO 0: LIMPIEZA ---
import os
import time
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By

os.system("pkill -9 chrome")
os.system("pkill -9 chromedriver")
print(" Limpieza lista")

# --- VARIABLES ---
NOMBRE_GRUPO = "AgroTech"
datos_finales = []
driver = None

# --- CONFIGURACIÓN CHROME ---
options = Options()
options.binary_location = "/usr/bin/google-chrome"

options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920,1080")

try:
    driver = webdriver.Chrome(options=options)
    print(" Navegador iniciado")

    # --- PASO 1: IR A ENAP ---
    driver.get("https://www.enap.cl/tabla-de-precios-de-paridad-historico")

    print(" Cargando página...")
    time.sleep(8)

    input(" Presiona ENTER para continuar...")

    # --- PASO 2: OBTENER TABLA ---
    filas = driver.find_elements(By.CSS_SELECTOR, "table tbody tr")

    print(f" Filas encontradas: {len(filas)}")

    for fila in filas:
        try:
            columnas = fila.find_elements(By.TAG_NAME, "td")

            # Ajusta según estructura real
            if len(columnas) >= 5:
                fecha = columnas[0].text
                diesel = columnas[4].text   #  DIÉSEL (posición típica)

                # Filtrar solo valores válidos
                if diesel != "":
                    datos_finales.append({
                        "fecha": fecha,
                        "diesel": diesel,
                        "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S"),
                        "grupo": NOMBRE_GRUPO
                    })

        except:
            continue

    print(f" Registros de diésel: {len(datos_finales)}")

except Exception as e:
    print(f" Error: {e}")

finally:
    if driver:
        driver.quit()

# --- PASO 3: GUARDAR EN MONGODB ---
try:
    client = MongoClient("database", 27017, serverSelectionTimeoutMS=5000)
    db = client["Energia"]
    coleccion = db["Diesel_ENAP"]

    if datos_finales:
        for d in datos_finales:
            v = str(d["diesel"]).replace(".", "").replace(",", "").strip()
            d["diesel"] = float(v) if v.isdigit() else 0.0

        coleccion.insert_many(datos_finales)
        print(" Datos guardados en MongoDB")
    else:
        print(" No se encontraron datos")

except Exception as e:
    print(f" Error MongoDB: {e}")

 Limpieza lista
 Navegador iniciado
 Cargando página...


 Presiona ENTER para continuar... 


 Filas encontradas: 379
 Registros de diésel: 377
 Datos guardados en MongoDB
